In [1]:
!pip -q install gradio pandas numpy plotly

In [2]:

import random
import numpy as np
import pandas as pd
import plotly.express as px
import gradio as gr

# ==========================================================
# PROJECT 8: AI SUPPLY CHAIN DISRUPTION & RISK MITIGATION
# Synthetic Toyota-style manufacturing demonstration only.
# NOT actual Toyota supplier or operational data.
# ==========================================================

random.seed(42)
np.random.seed(42)

SUPPLIERS = [
    "Supplier Alpha", "Supplier Beta", "Supplier Gamma",
    "Supplier Delta", "Supplier Epsilon", "Supplier Zeta"
]

MATERIALS = [
    "Engine Components", "Electronic Chips", "Steel Sheets",
    "Battery Modules", "Brake Components", "Tires",
    "Wiring Harnesses", "Cooling Systems"
]

TRANSPORT = ["Road", "Rail", "Air", "Sea"]
PLANTS = ["Plant A", "Plant B", "Plant C"]
PRIORITIES = ["Critical", "High", "Medium"]

# ----------------------------------------------------------
# 1. GENERATE SYNTHETIC SUPPLY CHAIN DATA
# ----------------------------------------------------------

rows = []

for i in range(1, 251):
    supplier = random.choice(SUPPLIERS)
    material = random.choice(MATERIALS)

    reliability = random.randint(65, 99)
    expected_lead = random.randint(3, 15)
    actual_lead = max(
        1, expected_lead + random.randint(-3, 10)
    )
    inventory_days = random.randint(1, 35)
    demand_change = random.randint(-10, 50)
    transport_delay = random.randint(0, 8)
    production_urgency = random.choice(PRIORITIES)
    order_quantity = random.randint(100, 2000)
    plant = random.choice(PLANTS)
    mode = random.choice(TRANSPORT)

    rows.append({
        "Order_ID": f"SC{i:04d}",
        "Supplier": supplier,
        "Material": material,
        "Plant": plant,
        "Transport_Mode": mode,
        "Supplier_Reliability_%": reliability,
        "Expected_Lead_Days": expected_lead,
        "Actual_Lead_Days": actual_lead,
        "Inventory_Cover_Days": inventory_days,
        "Demand_Increase_%": demand_change,
        "Transport_Delay_Days": transport_delay,
        "Production_Priority": production_urgency,
        "Order_Quantity": order_quantity
    })

df = pd.DataFrame(rows)

# ----------------------------------------------------------
# 2. RISK SCORING ENGINE
# Transparent rules make the agent explainable.
# ----------------------------------------------------------

def calculate_risk(data):
    data = data.copy()

    supplier_delay = (
        data["Actual_Lead_Days"] -
        data["Expected_Lead_Days"]
    ).clip(lower=0)

    # Individual risk indicators, each scaled from 0 to 100
    data["Supplier_Risk"] = (
        (100 - data["Supplier_Reliability_%"]) * 1.8
        + supplier_delay * 5
    ).clip(0, 100)

    data["Inventory_Risk"] = (
        (14 - data["Inventory_Cover_Days"]) * 7
        + data["Demand_Increase_%"] * 0.6
    ).clip(0, 100)

    data["Transport_Risk"] = (
        data["Transport_Delay_Days"] * 12
    ).clip(0, 100)

    data["Production_Risk"] = data["Production_Priority"].map({
        "Critical": 100,
        "High": 70,
        "Medium": 40
    })

    # Weighted overall score
    data["Risk_Score"] = (
        data["Supplier_Risk"] * 0.30
        + data["Inventory_Risk"] * 0.35
        + data["Transport_Risk"] * 0.20
        + data["Production_Risk"] * 0.15
    ).round(1)

    def risk_label(score):
        if score >= 65:
            return "High"
        elif score >= 35:
            return "Medium"
        return "Low"

    data["Risk_Level"] = data["Risk_Score"].apply(risk_label)

    # Identify the main risk driver for each order
    risk_columns = {
        "Supplier reliability / lead time": "Supplier_Risk",
        "Inventory shortage / demand increase": "Inventory_Risk",
        "Transportation delay": "Transport_Risk",
        "Production criticality": "Production_Risk"
    }

    reverse_map = {v: k for k, v in risk_columns.items()}
    data["Main_Risk_Driver"] = data[
        list(risk_columns.values())
    ].idxmax(axis=1).map(reverse_map)

    return data

scored_df = calculate_risk(df)

# ----------------------------------------------------------
# 3. MITIGATION RECOMMENDATION AGENT
# ----------------------------------------------------------

def recommend_action(row):
    actions = []

    if row["Inventory_Risk"] >= 50:
        actions.append(
            "Expedite replenishment; review safety stock"
        )

    if row["Supplier_Risk"] >= 50:
        actions.append(
            "Contact supplier; assess qualified backup source"
        )

    if row["Transport_Risk"] >= 35:
        actions.append(
            "Track shipment; evaluate alternative route"
        )

    if row["Production_Priority"] == "Critical":
        actions.append(
            "Prioritize material allocation to production"
        )

    if row["Demand_Increase_%"] >= 25:
        actions.append(
            "Review demand forecast and purchase quantities"
        )

    if row["Risk_Level"] == "Low" and not actions:
        actions.append(
            "Continue routine monitoring and supplier follow-up"
        )

    return " | ".join(dict.fromkeys(actions))

scored_df["Recommended_Action"] = scored_df.apply(
    recommend_action, axis=1
)

# ----------------------------------------------------------
# 4. INTERACTIVE ANALYSIS FUNCTION
# ----------------------------------------------------------

def run_agent(risk_filter, plant_filter, supplier_filter):
    data = scored_df.copy()

    if risk_filter != "All":
        data = data[data["Risk_Level"] == risk_filter]

    if plant_filter != "All Plants":
        data = data[data["Plant"] == plant_filter]

    if supplier_filter != "All Suppliers":
        data = data[data["Supplier"] == supplier_filter]

    total = len(data)

    if total == 0:
        empty = pd.DataFrame()
        fig = px.bar(title="No orders match the selected filters")
        return (
            "<h3>No matching orders</h3>",
            empty, empty, fig
        )

    high = int((data["Risk_Level"] == "High").sum())
    medium = int((data["Risk_Level"] == "Medium").sum())
    avg_risk = data["Risk_Score"].mean()
    shortage = int((data["Inventory_Risk"] >= 50).sum())
    supplier_issues = int((data["Supplier_Risk"] >= 50).sum())
    transport_issues = int((data["Transport_Risk"] >= 35).sum())

    summary = f"""
    <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));
    gap:12px;font-family:Arial,sans-serif;">
      <div style="background:#e9f2ff;padding:16px;border-radius:12px;">
        <div style="color:#334155;">Orders Analyzed</div>
        <h2 style="color:#16345c;">{total}</h2>
      </div>
      <div style="background:#fee2e2;padding:16px;border-radius:12px;">
        <div style="color:#7f1d1d;">High-Risk Orders</div>
        <h2 style="color:#991b1b;">{high}</h2>
      </div>
      <div style="background:#fef3c7;padding:16px;border-radius:12px;">
        <div style="color:#78350f;">Medium-Risk Orders</div>
        <h2 style="color:#92400e;">{medium}</h2>
      </div>
      <div style="background:#e0f2fe;padding:16px;border-radius:12px;">
        <div style="color:#075985;">Average Risk Score</div>
        <h2 style="color:#075985;">{avg_risk:.1f}/100</h2>
      </div>
      <div style="background:#f3e8ff;padding:16px;border-radius:12px;">
        <div style="color:#581c87;">Inventory Risk Alerts</div>
        <h2 style="color:#6b21a8;">{shortage}</h2>
      </div>
      <div style="background:#dcfce7;padding:16px;border-radius:12px;">
        <div style="color:#14532d;">Supplier / Transport Alerts</div>
        <h2 style="color:#166534;">{supplier_issues + transport_issues}</h2>
      </div>
    </div>
    <p style="margin-top:12px;padding:12px;background:#fff7ed;
    border-radius:8px;color:#9a3412;">
    Demonstration only: synthetic data and rule-based risk scores.
    These scores are not calibrated predictions of actual disruptions.
    </p>
    """

    # Risk distribution chart
    chart_data = (
        data.groupby(["Material", "Risk_Level"])
        .size().reset_index(name="Orders")
    )

    fig = px.bar(
        chart_data,
        x="Material",
        y="Orders",
        color="Risk_Level",
        barmode="group",
        title="Supply Chain Risk by Material",
        color_discrete_map={
            "Low": "#16a34a",
            "Medium": "#f59e0b",
            "High": "#dc2626"
        }
    )
    fig.update_layout(
        template="plotly_white",
        xaxis_tickangle=-25,
        height=450
    )

    # Prioritize highest risk orders
    alerts = data.sort_values(
        "Risk_Score", ascending=False
    )[[
        "Order_ID", "Supplier", "Material", "Plant",
        "Risk_Score", "Risk_Level", "Main_Risk_Driver",
        "Recommended_Action"
    ]].head(30)

    # Detailed operational data
    detail = data.sort_values(
        "Risk_Score", ascending=False
    )[[
        "Order_ID", "Supplier", "Material", "Plant",
        "Supplier_Reliability_%", "Expected_Lead_Days",
        "Actual_Lead_Days", "Inventory_Cover_Days",
        "Demand_Increase_%", "Transport_Mode",
        "Transport_Delay_Days", "Production_Priority",
        "Supplier_Risk", "Inventory_Risk",
        "Transport_Risk", "Risk_Score", "Risk_Level"
    ]]

    return summary, alerts, detail, fig

# ----------------------------------------------------------
# 5. GRADIO INTERACTIVE DASHBOARD
# ----------------------------------------------------------

CSS = """
.gradio-container {max-width:1450px !important;}
.hero {
 background:linear-gradient(120deg,#142b4a,#0f766e);
 color:white;padding:24px;border-radius:14px;margin-bottom:14px;
}
.hero h1 {color:white;margin:0 0 8px 0;}
.hero p {color:#e2e8f0;margin:0;}
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="teal"),
    css=CSS,
    title="Supply Chain Disruption Agent"
) as demo:

    gr.HTML("""
    <div class="hero">
      <h1>Manufacturing Supply Chain Intelligence Agent</h1>
      <p>Supplier monitoring • Inventory risk • Transport delays • Mitigation planning</p>
      <p style="margin-top:8px;"><b>Synthetic Toyota-style scenario — not actual Toyota data.</b></p>
    </div>
    """)

    with gr.Row():
        risk_filter = gr.Dropdown(
            ["All", "High", "Medium", "Low"],
            value="All", label="Risk level"
        )
        plant_filter = gr.Dropdown(
            ["All Plants"] + PLANTS,
            value="All Plants", label="Manufacturing plant"
        )
        supplier_filter = gr.Dropdown(
            ["All Suppliers"] + SUPPLIERS,
            value="All Suppliers", label="Supplier"
        )
        run_button = gr.Button(
            "Analyze Supply Chain", variant="primary"
        )

    summary_out = gr.HTML()

    with gr.Tabs():
        with gr.Tab("Risk Overview"):
            risk_chart = gr.Plot(label="Material risk distribution")
        with gr.Tab("Priority Alerts & Actions"):
            alerts_out = gr.Dataframe(
                label="Highest-risk orders and mitigation actions",
                interactive=False, wrap=True
            )
        with gr.Tab("Detailed Risk Analysis"):
            detail_out = gr.Dataframe(
                label="Supplier, inventory, transport and risk indicators",
                interactive=False, wrap=True
            )
        with gr.Tab("Methodology"):
            gr.Markdown("""
            ### How the agent works

            1. Generates synthetic supplier, inventory, transportation,
               demand, and production records.
            2. Calculates supplier risk from reliability and lead-time delays.
            3. Calculates inventory risk from days of inventory cover and
               changes in demand.
            4. Calculates transportation risk from shipment delays.
            5. Combines risk indicators into a weighted score from 0 to 100.
            6. Labels each order Low, Medium, or High risk.
            7. Selects the main risk driver and proposes mitigation actions.

            **Important limitation:** This version is an explainable,
            rule-based prototype. It does not train a forecasting model or
            validate risk against historical disruption outcomes. Scores
            illustrate how a system could prioritize investigation; they are
            not real-world calibrated probabilities.
            """)

    outputs = [
        summary_out, alerts_out, detail_out, risk_chart
    ]

    run_button.click(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    risk_filter.change(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    plant_filter.change(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    supplier_filter.change(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    demo.load(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )

demo.launch(share=True)


/tmp/ipykernel_3162/3255616178.py:314: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6fccdd2045e5fbbf7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:

import random
import numpy as np
import pandas as pd
import plotly.express as px
import gradio as gr

# ==========================================================
# PROJECT 8: AI SUPPLY CHAIN DISRUPTION & RISK MITIGATION
# Synthetic Toyota-style manufacturing demonstration only.
# NOT actual Toyota supplier or operational data.
# ==========================================================

random.seed(42)
np.random.seed(42)

SUPPLIERS = [
    "Supplier Alpha", "Supplier Beta", "Supplier Gamma",
    "Supplier Delta", "Supplier Epsilon", "Supplier Zeta"
]

MATERIALS = [
    "Engine Components", "Electronic Chips", "Steel Sheets",
    "Battery Modules", "Brake Components", "Tires",
    "Wiring Harnesses", "Cooling Systems"
]

TRANSPORT = ["Road", "Rail", "Air", "Sea"]
PLANTS = ["Plant A", "Plant B", "Plant C"]
PRIORITIES = ["Critical", "High", "Medium"]

# ----------------------------------------------------------
# 1. GENERATE SYNTHETIC SUPPLY CHAIN DATA
# ----------------------------------------------------------

rows = []

for i in range(1, 251):
    supplier = random.choice(SUPPLIERS)
    material = random.choice(MATERIALS)

    reliability = random.randint(65, 99)
    expected_lead = random.randint(3, 15)
    actual_lead = max(
        1, expected_lead + random.randint(-3, 10)
    )
    inventory_days = random.randint(1, 35)
    demand_change = random.randint(-10, 50)
    transport_delay = random.randint(0, 8)
    production_urgency = random.choice(PRIORITIES)
    order_quantity = random.randint(100, 2000)
    plant = random.choice(PLANTS)
    mode = random.choice(TRANSPORT)

    rows.append({
        "Order_ID": f"SC{i:04d}",
        "Supplier": supplier,
        "Material": material,
        "Plant": plant,
        "Transport_Mode": mode,
        "Supplier_Reliability_%": reliability,
        "Expected_Lead_Days": expected_lead,
        "Actual_Lead_Days": actual_lead,
        "Inventory_Cover_Days": inventory_days,
        "Demand_Increase_%": demand_change,
        "Transport_Delay_Days": transport_delay,
        "Production_Priority": production_urgency,
        "Order_Quantity": order_quantity
    })

df = pd.DataFrame(rows)

# ----------------------------------------------------------
# 2. RISK SCORING ENGINE
# Transparent rules make the agent explainable.
# ----------------------------------------------------------

def calculate_risk(data):
    data = data.copy()

    supplier_delay = (
        data["Actual_Lead_Days"] -
        data["Expected_Lead_Days"]
    ).clip(lower=0)

    # Individual risk indicators, each scaled from 0 to 100
    data["Supplier_Risk"] = (
        (100 - data["Supplier_Reliability_%"]) * 1.8
        + supplier_delay * 5
    ).clip(0, 100)

    data["Inventory_Risk"] = (
        (14 - data["Inventory_Cover_Days"]) * 7
        + data["Demand_Increase_%"] * 0.6
    ).clip(0, 100)

    data["Transport_Risk"] = (
        data["Transport_Delay_Days"] * 12
    ).clip(0, 100)

    data["Production_Risk"] = data["Production_Priority"].map({
        "Critical": 100,
        "High": 70,
        "Medium": 40
    })

    # Weighted overall score
    data["Risk_Score"] = (
        data["Supplier_Risk"] * 0.30
        + data["Inventory_Risk"] * 0.35
        + data["Transport_Risk"] * 0.20
        + data["Production_Risk"] * 0.15
    ).round(1)

    def risk_label(score):
        if score >= 65:
            return "High"
        elif score >= 35:
            return "Medium"
        return "Low"

    data["Risk_Level"] = data["Risk_Score"].apply(risk_label)

    # Identify the main risk driver for each order
    risk_columns = {
        "Supplier reliability / lead time": "Supplier_Risk",
        "Inventory shortage / demand increase": "Inventory_Risk",
        "Transportation delay": "Transport_Risk",
        "Production criticality": "Production_Risk"
    }

    reverse_map = {v: k for k, v in risk_columns.items()}
    data["Main_Risk_Driver"] = data[
        list(risk_columns.values())
    ].idxmax(axis=1).map(reverse_map)

    return data

scored_df = calculate_risk(df)

# ----------------------------------------------------------
# 3. MITIGATION RECOMMENDATION AGENT
# ----------------------------------------------------------

def recommend_action(row):
    actions = []

    if row["Inventory_Risk"] >= 50:
        actions.append(
            "Expedite replenishment; review safety stock"
        )

    if row["Supplier_Risk"] >= 50:
        actions.append(
            "Contact supplier; assess qualified backup source"
        )

    if row["Transport_Risk"] >= 35:
        actions.append(
            "Track shipment; evaluate alternative route"
        )

    if row["Production_Priority"] == "Critical":
        actions.append(
            "Prioritize material allocation to production"
        )

    if row["Demand_Increase_%"] >= 25:
        actions.append(
            "Review demand forecast and purchase quantities"
        )

    if row["Risk_Level"] == "Low" and not actions:
        actions.append(
            "Continue routine monitoring and supplier follow-up"
        )

    return " | ".join(dict.fromkeys(actions))

scored_df["Recommended_Action"] = scored_df.apply(
    recommend_action, axis=1
)

# ----------------------------------------------------------
# 4. INTERACTIVE ANALYSIS FUNCTION
# ----------------------------------------------------------

def run_agent(risk_filter, plant_filter, supplier_filter):
    data = scored_df.copy()

    if risk_filter != "All":
        data = data[data["Risk_Level"] == risk_filter]

    if plant_filter != "All Plants":
        data = data[data["Plant"] == plant_filter]

    if supplier_filter != "All Suppliers":
        data = data[data["Supplier"] == supplier_filter]

    total = len(data)

    if total == 0:
        empty = pd.DataFrame()
        fig = px.bar(title="No orders match the selected filters")
        return (
            "<h3>No matching orders</h3>",
            empty, empty, fig
        )

    high = int((data["Risk_Level"] == "High").sum())
    medium = int((data["Risk_Level"] == "Medium").sum())
    avg_risk = data["Risk_Score"].mean()
    shortage = int((data["Inventory_Risk"] >= 50).sum())
    supplier_issues = int((data["Supplier_Risk"] >= 50).sum())
    transport_issues = int((data["Transport_Risk"] >= 35).sum())

    summary = f"""
    <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));
    gap:12px;font-family:Arial,sans-serif;">
      <div style="background:#e9f2ff;padding:16px;border-radius:12px;">
        <div style="color:#334155;">Orders Analyzed</div>
        <h2 style="color:#16345c;">{total}</h2>
      </div>
      <div style="background:#fee2e2;padding:16px;border-radius:12px;">
        <div style="color:#7f1d1d;">High-Risk Orders</div>
        <h2 style="color:#991b1b;">{high}</h2>
      </div>
      <div style="background:#fef3c7;padding:16px;border-radius:12px;">
        <div style="color:#78350f;">Medium-Risk Orders</div>
        <h2 style="color:#92400e;">{medium}</h2>
      </div>
      <div style="background:#e0f2fe;padding:16px;border-radius:12px;">
        <div style="color:#075985;">Average Risk Score</div>
        <h2 style="color:#075985;">{avg_risk:.1f}/100</h2>
      </div>
      <div style="background:#f3e8ff;padding:16px;border-radius:12px;">
        <div style="color:#581c87;">Inventory Risk Alerts</div>
        <h2 style="color:#6b21a8;">{shortage}</h2>
      </div>
      <div style="background:#dcfce7;padding:16px;border-radius:12px;">
        <div style="color:#14532d;">Supplier / Transport Alerts</div>
        <h2 style="color:#166534;">{supplier_issues + transport_issues}</h2>
      </div>
    </div>
    <p style="margin-top:12px;padding:12px;background:#fff7ed;
    border-radius:8px;color:#9a3412;">
    Demonstration only: synthetic data and rule-based risk scores.
    These scores are not calibrated predictions of actual disruptions.
    </p>
    """

    # Risk distribution chart
    chart_data = (
        data.groupby(["Material", "Risk_Level"])
        .size().reset_index(name="Orders")
    )

    fig = px.bar(
        chart_data,
        x="Material",
        y="Orders",
        color="Risk_Level",
        barmode="group",
        title="Supply Chain Risk by Material",
        color_discrete_map={
            "Low": "#16a34a",
            "Medium": "#f59e0b",
            "High": "#dc2626"
        }
    )
    fig.update_layout(
        template="plotly_white",
        xaxis_tickangle=-25,
        height=450
    )

    # Prioritize highest risk orders
    alerts = data.sort_values(
        "Risk_Score", ascending=False
    )[[
        "Order_ID", "Supplier", "Material", "Plant",
        "Risk_Score", "Risk_Level", "Main_Risk_Driver",
        "Recommended_Action"
    ]].head(30)

    # Detailed operational data
    detail = data.sort_values(
        "Risk_Score", ascending=False
    )[[
        "Order_ID", "Supplier", "Material", "Plant",
        "Supplier_Reliability_%", "Expected_Lead_Days",
        "Actual_Lead_Days", "Inventory_Cover_Days",
        "Demand_Increase_%", "Transport_Mode",
        "Transport_Delay_Days", "Production_Priority",
        "Supplier_Risk", "Inventory_Risk",
        "Transport_Risk", "Risk_Score", "Risk_Level"
    ]]

    return summary, alerts, detail, fig

# ----------------------------------------------------------
# 5. GRADIO INTERACTIVE DASHBOARD
# ----------------------------------------------------------

CSS = """
.gradio-container {max-width:1450px !important;}
.hero {
 background:linear-gradient(120deg,#142b4a,#0f766e);
 color:white;padding:24px;border-radius:14px;margin-bottom:14px;
}
.hero h1 {color:white;margin:0 0 8px 0;}
.hero p {color:#e2e8f0;margin:0;}
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="teal"),
    css=CSS,
    title="Supply Chain Disruption Agent"
) as demo:

    gr.HTML("""
    <div class="hero">
      <h1>Manufacturing Supply Chain Intelligence Agent</h1>
      <p>Supplier monitoring • Inventory risk • Transport delays • Mitigation planning</p>
      <p style="margin-top:8px;"><b>Synthetic Toyota-style scenario — not actual Toyota data.</b></p>
    </div>
    """)

    with gr.Row():
        risk_filter = gr.Dropdown(
            ["All", "High", "Medium", "Low"],
            value="All", label="Risk level"
        )
        plant_filter = gr.Dropdown(
            ["All Plants"] + PLANTS,
            value="All Plants", label="Manufacturing plant"
        )
        supplier_filter = gr.Dropdown(
            ["All Suppliers"] + SUPPLIERS,
            value="All Suppliers", label="Supplier"
        )
        run_button = gr.Button(
            "Analyze Supply Chain", variant="primary"
        )

    summary_out = gr.HTML()

    with gr.Tabs():
        with gr.Tab("Risk Overview"):
            risk_chart = gr.Plot(label="Material risk distribution")
        with gr.Tab("Priority Alerts & Actions"):
            alerts_out = gr.Dataframe(
                label="Highest-risk orders and mitigation actions",
                interactive=False, wrap=True
            )
        with gr.Tab("Detailed Risk Analysis"):
            detail_out = gr.Dataframe(
                label="Supplier, inventory, transport and risk indicators",
                interactive=False, wrap=True
            )
        with gr.Tab("Methodology"):
            gr.Markdown("""
            ### How the agent works

            1. Generates synthetic supplier, inventory, transportation,
               demand, and production records.
            2. Calculates supplier risk from reliability and lead-time delays.
            3. Calculates inventory risk from days of inventory cover and
               changes in demand.
            4. Calculates transportation risk from shipment delays.
            5. Combines risk indicators into a weighted score from 0 to 100.
            6. Labels each order Low, Medium, or High risk.
            7. Selects the main risk driver and proposes mitigation actions.

            **Important limitation:** This version is an explainable,
            rule-based prototype. It does not train a forecasting model or
            validate risk against historical disruption outcomes. Scores
            illustrate how a system could prioritize investigation; they are
            not real-world calibrated probabilities.
            """)

    outputs = [
        summary_out, alerts_out, detail_out, risk_chart
    ]

    run_button.click(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    risk_filter.change(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    plant_filter.change(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    supplier_filter.change(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )
    demo.load(
        run_agent,
        inputs=[risk_filter, plant_filter, supplier_filter],
        outputs=outputs
    )

demo.launch(share=True)


/tmp/ipykernel_3162/3255616178.py:314: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8122d601aae4f3ce71.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
